### Naiver Bayes

[Video](https://youtu.be/YeJkWzrsoAI)

Der Naive Bayes Classifier wird meist für Klassifikationsaufgaben eingesetzt, bei denen die Daten text- oder kategorienbasiert sind und eine schnelle, einfache Lösung gefragt ist. Anwendungsbereiche sind:


- Spam-Filter (E-Mail als „Spam“ oder „Nicht-Spam“ erkennen)
- Sentiment-Analyse (z. B. „positiv“ / „negativ“ in Produktbewertungen)
- Themenklassifikation (z. B. Nachricht als „Sport“, „Politik“, „Wirtschaft“)


Naive Bayes ist schnell und benötigt wenig Trainingsdaten und ist einfach zu implementieren.

**Beispiel**:


<img src='spam.png' width='602'>

**Beispiel** mit scikit-learn:

Der Klassifier soll herausfinden, ob ein Text von einem Hund oder einer Katze handelt oder Neutral ist.

In [1]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
import pandas as pd
import math

texte = [
    "Katzen sind niedlich",
    "Hunde sind treu",
    "Katzen mögen Milch",
    "Hunde spielen gern",
    "Milch ist lecker",
    "Ich liebe Hunde"
]
labels = ["Katze", "Hund", "Katze", "Hund", "Neutral", "Hund"]

In [2]:
print("Trainingsdaten:")
for t, l in zip(texte, labels):
    print(f"  '{t}' -> {l}")

Trainingsdaten:
  'Katzen sind niedlich' -> Katze
  'Hunde sind treu' -> Hund
  'Katzen mögen Milch' -> Katze
  'Hunde spielen gern' -> Hund
  'Milch ist lecker' -> Neutral
  'Ich liebe Hunde' -> Hund


Jedes Wort bekommt einen Index zugewiesen

In [3]:
# Bag-of-Words erstellen
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texte)
vectorizer.get_feature_names_out()

array(['gern', 'hunde', 'ich', 'ist', 'katzen', 'lecker', 'liebe',
       'milch', 'mögen', 'niedlich', 'sind', 'spielen', 'treu'],
      dtype=object)

In [4]:
df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df

,gern,hunde,ich,ist,katzen,lecker,liebe,milch,mögen,niedlich,sind,spielen,treu
0,0,0,0,0,1,0,0,0,0,1,1,0,0
1,0,1,0,0,0,0,0,0,0,0,1,0,1
2,0,0,0,0,1,0,0,1,1,0,0,0,0
3,1,1,0,0,0,0,0,0,0,0,0,1,0
4,0,0,0,1,0,1,0,1,0,0,0,0,0
5,0,1,1,0,0,0,1,0,0,0,0,0,0


In [13]:
model = MultinomialNB()
model.fit(X, labels)

# Klassen-Wahrscheinlichkeiten anzeigen
print("Klassen im Modell:", model.classes_)
print("Klassen-Prior-Wahrscheinlichkeiten:", model.class_log_prior_)

Klassen im Modell: ['Hund' 'Katze' 'Neutral']
Klassen-Prior-Wahrscheinlichkeiten: [-0.69314718 -1.09861229 -1.79175947]


Berechnung der Klassen-Prior-Wahrscheinlichkeiten:  

In [16]:
# Hund kommt in 3 von 6 Fällen vor
math.log(1/6)

-1.791759469228055

In [21]:
# Bedingte Wahrscheinlichkeiten: Wort gegeben Klasse
bed_wk = pd.DataFrame(model.feature_log_prob_, 
                      index=model.classes_, 
                      columns=vectorizer.get_feature_names_out())
bed_wk

,gern,hunde,ich,ist,katzen,lecker,liebe,milch,mögen,niedlich,sind,spielen,treu
Hund,-2.397895,-1.704748,-2.397895,-3.091042,-3.091042,-3.091042,-2.397895,-3.091042,-3.091042,-3.091042,-2.397895,-2.397895,-2.397895
Katze,-2.944439,-2.944439,-2.944439,-2.944439,-1.845827,-2.944439,-2.944439,-2.251292,-2.251292,-2.251292,-2.251292,-2.944439,-2.944439
Neutral,-2.772589,-2.772589,-2.772589,-2.079442,-2.772589,-2.079442,-2.772589,-2.079442,-2.772589,-2.772589,-2.772589,-2.772589,-2.772589


Berechnung der *feature_log_prob_* nach folgender Formel

$P(Wort | Klasse) = \dfrac{\text{Häufigkeit} + 1}{\text{Gesamtwortzahl in Klasse} + \text{|Vokabular|}}$

In [22]:
# P(milch|katze)
math.log((1 + 1)/(6 + 13))

-2.2512917986064953

In [23]:
#P(hunde|treu)
math.log((1 + 1)/(9 + 13))

-2.3978952727983707

In [24]:
# Neue Texte testen
neue_texte = ["Katzen trinken Milch", "Hunde sind toll", "Milch schmeckt"]
X_neu = vectorizer.transform(neue_texte)
df_neu = pd.DataFrame(X_neu.toarray(), columns=vectorizer.get_feature_names_out())
df_neu

,gern,hunde,ich,ist,katzen,lecker,liebe,milch,mögen,niedlich,sind,spielen,treu
0,0,0,0,0,1,0,0,1,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,1,0,0
2,0,0,0,0,0,0,0,1,0,0,0,0,0


In [25]:
vorhersagen = model.predict(X_neu)
for text, label in zip(neue_texte, vorhersagen):
    print(f"  '{text}' -> {label}" )

  'Katzen trinken Milch' -> Katze
  'Hunde sind toll' -> Hund
  'Milch schmeckt' -> Katze


In [28]:
model.predict_proba(X_neu)

array([[0.13117683, 0.70348571, 0.16533746],
       [0.76791385, 0.17159294, 0.06049321],
       [0.28897338, 0.44613435, 0.26489227]])

###

#### Exkurs: Warum Logarithmus?
Warum wird bei der Berechnung der Wahrscheinlichkeiten der Logrithmus benutzt?

In [18]:
import math

# Wir simulieren:
# - Klasse A: Wahrscheinlichkeit jedes Features = 0.01
# - Klasse B: Wahrscheinlichkeit jedes Features = 0.02
# - Wir wollen die Klasse mit der größeren Gesamtwahrscheinlichkeit finden

p_A = 0.01
p_B = 0.02
n = 1000  # Anzahl der Features/Wörter

print("=== OHNE LOGARITHMUS ===")

prob_A = (p_A) ** n
prob_B = (p_B) ** n
print("P(A) =", prob_A)
print("P(B) =", prob_B)

=== OHNE LOGARITHMUS ===
P(A) = 0.0
P(B) = 0.0


In [19]:
print("\n=== MIT LOGARITHMUS ===")
log_A = n * math.log(p_A)
log_B = n * math.log(p_B)
print("log(P(A)) =", log_A)
print("log(P(B)) =", log_B)


=== MIT LOGARITHMUS ===
log(P(A)) = -4605.170185988091
log(P(B)) = -3912.023005428146
